### Importing libraries and dataset

In [1]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_param_importances
)
from xgboost import XGBRegressor
from sklearn.svm import SVR
import pandas as pd
import numpy as np
import dagshub
import mlflow
import optuna
import tqdm

c:\Users\Lenovo\miniconda3\envs\uber\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn import set_config

set_config(transform_output="pandas")

In [ ]:
# set the dagshub tracking server

mlflow.set_tracking_uri("https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow")
dagshub.init(repo_owner='mrvivekkumar7171', repo_name='uber-demand-prediction', mlflow=True)

In [5]:
# load the training and test data

train_data_path = "../data/processed/train.csv"
test_data_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")
test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")
train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,14.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


### Preparing the dataset

In [6]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [7]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [8]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])
y_train = train_df["total_pickups"]

In [9]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [10]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])
y_test = test_df["total_pickups"]

In [11]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,39.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,37.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,41.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,38.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,35.0,1


### Hyperparameter tuning and model selection

In [12]:
# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["region", "day_of_week"])
], remainder="passthrough", n_jobs=-1)

In [13]:
encoder

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``

In [14]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [15]:
# set the experiment

mlflow.set_experiment("Model Selection")

<Experiment: artifact_location='mlflow-artifacts:/b6b4d8f3b62643e39281e1bb68a431be', creation_time=1782047742355, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1782047742355, lifecycle_stage='active', name='Model Selection', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)

        # we can also use time series model for train the model like ARIMA, SARIMA etc.
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf", 10, 100, step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf", 3, 10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, max_depth=max_depth_rf, random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb", 10, 100, step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb", 1e-4, 1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, learning_rate=learning_rate_gb, random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb", 10, 100, step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb", 1e-4, 1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb", 3, 10)
            model = XGBRegressor(n_estimators=n_estimators_xgb, learning_rate=learning_rate_xgb, max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name", model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded, y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE", loss)
        return loss

In [17]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-06-21 19:30:26,796] A new study created in memory with name: model_selection


🏃 View run zealous-doe-162 at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0/runs/8aa08b01011541c2a9e7cee5f5b61db4
🧪 View experiment at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-21 19:30:31,985] Trial 2 finished with value: 0.07934790285463093 and parameters: {'model_name': 'LR'}. Best is trial 2 with value: 0.07934790285463093.


🏃 View run auspicious-bat-82 at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0/runs/56be401f72fe47d6ac1fc18da9716b02
🧪 View experiment at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-21 19:30:32,386] Trial 3 finished with value: 0.07934790285463093 and parameters: {'model_name': 'LR'}. Best is trial 2 with value: 0.07934790285463093.


...

🏃 View run fearless-stag-686 at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0/runs/ecb64389196d45eaaf73716d01a8942d
🧪 View experiment at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-21 19:40:59,380] Trial 47 finished with value: 0.07934790285463093 and parameters: {'model_name': 'LR'}. Best is trial 2 with value: 0.07934790285463093.


🏃 View run angry-hen-555 at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0/runs/550a7938cece481ca8b159b98ca43223
🧪 View experiment at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-21 19:41:16,267] Trial 49 finished with value: 5.685380787601503 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 50, 'learning_rate_gb': 0.0031014818513940556}. Best is trial 2 with value: 0.07934790285463093.


🏃 View run best_model at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0/runs/8c68f900b9e1414c80f80413e5d4ec91
🧪 View experiment at: https://dagshub.com/mrvivekkumar7171/uber-demand-prediction.mlflow/#/experiments/0


In [18]:
# best value

study.best_value

0.07934790285463093

In [19]:
# best parameters

study.best_params

{'model_name': 'LR'}

In [20]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
LR      28
XGBR     9
RF       7
GBR      6
Name: count, dtype: int64

We can say that LR is our bast model with around 8% error. Mostly LR is used means LR is stable and best for the problem.

In [21]:
plot_optimization_history(study)

This shows that our model got optimized after 2 trails only

In [22]:
plot_parallel_coordinate(study, params=["model_name"])

LR is constantly giving low error. Thus training the best model alone again.

In [23]:
# train the linear regression model

lr = LinearRegression()

lr.fit(X_train_encoded, y_train)

# get predictions
y_pred_train = lr.predict(X_train_encoded) 
y_pred_test = lr.predict(X_test_encoded)

# loss

mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

print("The training error is ", mape_train)
print("The test error is ", mape_test)

The training error is  0.08778013304566609
The test error is  0.07934790285463093


Again, we are getting the same error of 8%

In [24]:
# weights of the linear regression model
lr.coef_

array([-2.33737604,  0.71512405, -0.55601505, -1.25311068, -3.20463231,
       -0.86685973, -2.79925402, -3.62516859,  0.41386463, -2.9376376 ,
       -1.97624678, -3.75050442,  0.51806283, -2.54033388, -2.43297463,
        0.47632075,  0.61254786, -4.7417372 , -2.03077217, -1.26960984,
       -4.03690273, -2.08863167, -1.0414428 ,  0.73561736, -0.99999442,
       -0.85944985, -2.43098478,  0.67112238,  0.57385071, -0.11719951,
       -0.28045898, -0.37180749, -0.5238324 , -0.4233113 , -0.34045774,
       -0.54170892, -0.36264553, -0.2493965 , -0.31905518,  2.4912456 ])

### Ridge

In [25]:
def tune_ridge(trial):
    # hyperparameter space
    alpha = trial.suggest_float("alpha", 30, 100)
    
    # make the model object
    ridge = Ridge(alpha=alpha, random_state=42)
    
    # train the model
    ridge.fit(X_train_encoded, y_train)
    
    # get predictions
    y_pred = ridge.predict(X_test_encoded)
    
    # calculate loss
    loss = mean_absolute_percentage_error(y_test, y_pred)

    return loss

In [26]:
# create study

study = optuna.create_study(study_name="tune_model", direction="minimize")

[I 2026-06-21 19:41:20,167] A new study created in memory with name: tune_model


In [27]:
# optimize

study.optimize(func=tune_ridge, n_trials=100, n_jobs=-1, show_progress_bar=True)

Best trial: 3. Best value: 0.0791701:   3%|▎         | 3/100 [00:00<00:24,  3.98it/s]

[I 2026-06-21 19:41:20,411] Trial 0 finished with value: 0.07926979411635493 and parameters: {'alpha': 32.90993180270842}. Best is trial 0 with value: 0.07926979411635493.
[I 2026-06-21 19:41:20,427] Trial 1 finished with value: 0.07919400253867599 and parameters: {'alpha': 77.66603787869215}. Best is trial 1 with value: 0.07919400253867599.
[I 2026-06-21 19:41:20,436] Trial 2 finished with value: 0.0792432695613644 and parameters: {'alpha': 46.7747675300711}. Best is trial 1 with value: 0.07919400253867599.
[I 2026-06-21 19:41:20,437] Trial 3 finished with value: 0.07917014311112226 and parameters: {'alpha': 95.12849969750454}. Best is trial 3 with value: 0.07917014311112226.


Best trial: 3. Best value: 0.0791701:   7%|▋         | 7/100 [00:00<00:07, 12.00it/s]

[I 2026-06-21 19:41:20,638] Trial 4 finished with value: 0.07917963078674678 and parameters: {'alpha': 87.94074062128922}. Best is trial 3 with value: 0.07917014311112226.
[I 2026-06-21 19:41:20,649] Trial 5 finished with value: 0.07921005647132995 and parameters: {'alpha': 66.84769146142605}. Best is trial 3 with value: 0.07917014311112226.
[I 2026-06-21 19:41:20,711] Trial 6 finished with value: 0.07917551017505223 and parameters: {'alpha': 91.0181748013951}. Best is trial 3 with value: 0.07917014311112226.
[I 2026-06-21 19:41:20,747] Trial 7 finished with value: 0.07920982506381359 and parameters: {'alpha': 66.99937658254683}. Best is trial 3 with value: 0.07917014311112226.


Best trial: 11. Best value: 0.0791679:  11%|█         | 11/100 [00:00<00:06, 13.32it/s]

...

[I 2026-06-21 19:41:26,315] Trial 87 finished with value: 0.07926324552652664 and parameters: {'alpha': 36.198735945855006}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,342] Trial 88 finished with value: 0.07917483474577436 and parameters: {'alpha': 91.52869929379769}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,397] Trial 89 finished with value: 0.07916618317810971 and parameters: {'alpha': 98.28277085183643}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,456] Trial 90 finished with value: 0.07917520395242479 and parameters: {'alpha': 91.24935153172407}. Best is trial 57 with value: 0.07916406791899547.


Best trial: 57. Best value: 0.0791641:  94%|█████████▍| 94/100 [00:06<00:00, 12.21it/s]

[I 2026-06-21 19:41:26,597] Trial 91 finished with value: 0.07917538328310032 and parameters: {'alpha': 91.11396347735457}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,660] Trial 93 finished with value: 0.07916413887225207 and parameters: {'alpha': 99.94027805473607}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,666] Trial 92 finished with value: 0.07916673952019775 and parameters: {'alpha': 97.83451942663798}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,721] Trial 94 finished with value: 0.0791642899062411 and parameters: {'alpha': 99.81724999596211}. Best is trial 57 with value: 0.07916406791899547.


Best trial: 57. Best value: 0.0791641: 100%|██████████| 100/100 [00:06<00:00, 14.58it/s]

[I 2026-06-21 19:41:26,849] Trial 95 finished with value: 0.07916439304245243 and parameters: {'alpha': 99.73324511874397}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,928] Trial 96 finished with value: 0.07916416827639253 and parameters: {'alpha': 99.91632531239266}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,937] Trial 97 finished with value: 0.07916860720472464 and parameters: {'alpha': 96.34309427059972}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:26,994] Trial 98 finished with value: 0.07916408567942186 and parameters: {'alpha': 99.98361034220684}. Best is trial 57 with value: 0.07916406791899547.
[I 2026-06-21 19:41:27,039] Trial 99 finished with value: 0.07916908282967133 and parameters: {'alpha': 95.9654520304162}. Best is trial 57 with value: 0.07916406791899547.


In [28]:
# best parameters

study.best_params

{'alpha': 99.99807879059418}

In [29]:
# best value

study.best_value

0.07916406791899547

error is same as LR. Hence, we are going with LR as our final model.